In [2]:
import sys
from pathlib import Path

def find_project_root(markers=("src", "Data", "Notebooks")):
    current_path = Path.cwd().resolve()

    for path in [current_path] + list(current_path.parents):
        has_all_markers = all((path / marker).exists() for marker in markers)
        if has_all_markers:
            return path

    raise RuntimeError(f"Project root could not be found from: {current_path}")

project_root = find_project_root()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import paths as P

In [3]:
import json
import pandas as pd

json_files = list(P.decision_logs_raw_dir.rglob("*.json"))

print("Number of JSON files found:")
print(len(json_files))
print()

print("First 5 files:")
for file in json_files[:5]:
    print(file)

Number of JSON files found:
137

First 5 files:
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087643609_b1186f57813e18.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087849461_b8982278927ba8.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087894772_fbccd276826a48.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087905071_cea20dd6486ef8.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Lo

In [ ]:
import re

# find all json files inside all class folders under Raw
json_files = sorted(P.decision_logs_raw_dir.rglob("*.json"))

print(f"JSON files found: {len(json_files)}")
for file in json_files[:5]:
    print(file)

all_rows = []

def parse_folder_info(folder_name):
    folder_clean = folder_name.strip()

    module_match = re.search(r"(CT\d+)", folder_clean, flags=re.IGNORECASE)
    module_code = module_match.group(1).upper() if module_match else None

    folder_lower = folder_clean.lower()
    if "morning" in folder_lower:
        session_time = "Morning"
    elif "afternoon" in folder_lower:
        session_time = "Afternoon"
    else:
        session_time = "Not specified"

    return module_code, session_time

for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    source_folder = file.parent.name
    source_file = file.name
    module_code, session_time = parse_folder_info(source_folder)

    run_id = data.get("run_id")
    sent_at = data.get("sent_at")

    final_scores = data.get("finalScores", {})
    decisions = data.get("decisionLog", [])

    if not isinstance(decisions, list):
        decisions = []

    n_decisions = len(decisions)
    completed_run = n_decisions == 15
    ended_early = n_decisions < 15

    # if a file has no decisions, still keep one row with top-level fields
    if n_decisions == 0:
        row = {
            "source_folder": source_folder,
            "module_code": module_code,
            "session_time": session_time,
            "source_file": source_file,

            "run_id": run_id,
            "sent_at": sent_at,

            "finalScores_budget": final_scores.get("budget"),
            "finalScores_reputation": final_scores.get("reputation"),
            "finalScores_security": final_scores.get("security"),
            "finalScores_morale": final_scores.get("morale"),

            "decision_order": None,
            "decisionLog_scenario": None,
            "decisionLog_choice": None,
            "decisionLog_outcome": None,
            "decisionLog_timestamp": None,

            "decisionLog_effects_budget": None,
            "decisionLog_effects_reputation": None,
            "decisionLog_effects_security": None,
            "decisionLog_effects_morale": None,

            "n_decisions": n_decisions,
            "completed_run": completed_run,
            "ended_early": ended_early
        }
        all_rows.append(row)

    # normal case: one row per decision
    for i, d in enumerate(decisions, start=1):
        effects = d.get("effects", {})

        row = {
            "source_folder": source_folder,
            "module_code": module_code,
            "session_time": session_time,
            "source_file": source_file,

            "run_id": run_id,
            "sent_at": sent_at,

            "finalScores_budget": final_scores.get("budget"),
            "finalScores_reputation": final_scores.get("reputation"),
            "finalScores_security": final_scores.get("security"),
            "finalScores_morale": final_scores.get("morale"),

            "decision_order": i,
            "decisionLog_scenario": d.get("scenario"),
            "decisionLog_choice": d.get("choice"),
            "decisionLog_outcome": d.get("outcome"),
            "decisionLog_timestamp": d.get("timestamp"),

            "decisionLog_effects_budget": effects.get("budget"),
            "decisionLog_effects_reputation": effects.get("reputation"),
            "decisionLog_effects_security": effects.get("security"),
            "decisionLog_effects_morale": effects.get("morale"),

            "n_decisions": n_decisions,
            "completed_run": completed_run,
            "ended_early": ended_early
        }

        all_rows.append(row)

master_df = pd.DataFrame(all_rows)

master_file = P.decision_logs_interim_dir / "master_decision_logs.csv"
master_df.to_csv(master_file, index=False)

print("\nMaster dataset created:")
print(master_file)
print("Shape:", master_df.shape)

print("\nColumns:")
print(master_df.columns.tolist())

print("\nSource folders found:")
print(master_df["source_folder"].drop_duplicates().tolist())

print("\nCompleted vs ended early:")
print(master_df[["completed_run", "ended_early"]].drop_duplicates())

print("\nFirst 5 rows:")
print(master_df.head())

JSON files found: 137
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087643609_b1186f57813e18.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087849461_b8982278927ba8.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087894772_fbccd276826a48.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon decision logs\ISM_Decision_Log_run_1776087905071_cea20dd6486ef8.json
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Raw\CT4010 afternoon de

In [ ]:
#load the master dataset
master_file = P.decision_logs_interim_dir / "master_decision_logs.csv"
master_df = pd.read_csv(master_file)

print("Master dataset shape:")
print(master_df.shape)
master_df.head()

Master dataset shape:
(2024, 22)


,source_folder,module_code,session_time,source_file,run_id,sent_at,finalScores_budget,finalScores_reputation,finalScores_security,finalScores_morale,...,decisionLog_choice,decisionLog_outcome,decisionLog_timestamp,decisionLog_effects_budget,decisionLog_effects_reputation,decisionLog_effects_security,decisionLog_effects_morale,n_decisions,completed_run,ended_early
0,CT4010 afternoon decision logs,CT4010,Afternoon,ISM_Decision_Log_run_1776087643609_b1186f57813...,run_1776087643609_b1186f57813e18,2026-04-13T13:40:43.609Z,276000,125,178,113,...,Negotiate While Recovering,<strong>Security:</strong> remains stable beca...,2026-04-13T13:35:32.447Z,-120000,-18,0,-10,15,True,False
1,CT4010 afternoon decision logs,CT4010,Afternoon,ISM_Decision_Log_run_1776087643609_b1186f57813...,run_1776087643609_b1186f57813e18,2026-04-13T13:40:43.609Z,276000,125,178,113,...,Move to Third-Party Cloud,<strong>Security:</strong> increases because c...,2026-04-13T13:35:53.838Z,25000,2,4,1,15,True,False
2,CT4010 afternoon decision logs,CT4010,Afternoon,ISM_Decision_Log_run_1776087643609_b1186f57813...,run_1776087643609_b1186f57813e18,2026-04-13T13:40:43.609Z,276000,125,178,113,...,Phased Access Review,<strong>Security:</strong> increases moderatel...,2026-04-13T13:36:15.658Z,-22000,6,8,4,15,True,False
3,CT4010 afternoon decision logs,CT4010,Afternoon,ISM_Decision_Log_run_1776087643609_b1186f57813...,run_1776087643609_b1186f57813e18,2026-04-13T13:40:43.609Z,276000,125,178,113,...,Apply Temporary Mitigations,<strong>Security:</strong> increases moderatel...,2026-04-13T13:36:40.937Z,-30000,3,8,1,15,True,False
4,CT4010 afternoon decision logs,CT4010,Afternoon,ISM_Decision_Log_run_1776087643609_b1186f57813...,run_1776087643609_b1186f57813e18,2026-04-13T13:40:43.609Z,276000,125,178,113,...,Optional Training Modules,<strong>Security:</strong> increases slightly ...,2026-04-13T13:37:01.639Z,10000,1,2,1,15,True,False


In [6]:
#check columns and data types
print("Columns:")
print(master_df.columns.tolist())
print()

print("Data types:")
print(master_df.dtypes)

Columns:
['source_folder', 'module_code', 'session_time', 'source_file', 'run_id', 'sent_at', 'finalScores_budget', 'finalScores_reputation', 'finalScores_security', 'finalScores_morale', 'decision_order', 'decisionLog_scenario', 'decisionLog_choice', 'decisionLog_outcome', 'decisionLog_timestamp', 'decisionLog_effects_budget', 'decisionLog_effects_reputation', 'decisionLog_effects_security', 'decisionLog_effects_morale', 'n_decisions', 'completed_run', 'ended_early']

Data types:
source_folder                     object
module_code                       object
session_time                      object
source_file                       object
run_id                            object
sent_at                           object
finalScores_budget                 int64
finalScores_reputation             int64
finalScores_security               int64
finalScores_morale                 int64
decision_order                     int64
decisionLog_scenario              object
decisionLog_choice    

In [7]:
# Check for missing values
missing_summary = master_df.isna().sum().sort_values(ascending=False)

print("Missing values by column:")
print(missing_summary)

Missing values by column:
source_folder                     0
module_code                       0
session_time                      0
source_file                       0
run_id                            0
sent_at                           0
finalScores_budget                0
finalScores_reputation            0
finalScores_security              0
finalScores_morale                0
decision_order                    0
decisionLog_scenario              0
decisionLog_choice                0
decisionLog_outcome               0
decisionLog_timestamp             0
decisionLog_effects_budget        0
decisionLog_effects_reputation    0
decisionLog_effects_security      0
decisionLog_effects_morale        0
n_decisions                       0
completed_run                     0
ended_early                       0
dtype: int64


In [ ]:
# missing values in key columns
key_cols = [
    "source_folder",
    "module_code",
    "session_time",
    "source_file",
    "run_id",
    "sent_at",
    "decision_order",
    "decisionLog_scenario",
    "decisionLog_choice",
    "decisionLog_outcome",
    "decisionLog_timestamp",
    "decisionLog_effects_budget",
    "decisionLog_effects_reputation",
    "decisionLog_effects_security",
    "decisionLog_effects_morale"
]

print("Missing values in key columns:")
print(master_df[key_cols].isna().sum())

Missing values in key columns:
source_folder                     0
module_code                       0
session_time                      0
source_file                       0
run_id                            0
sent_at                           0
decision_order                    0
decisionLog_scenario              0
decisionLog_choice                0
decisionLog_outcome               0
decisionLog_timestamp             0
decisionLog_effects_budget        0
decisionLog_effects_reputation    0
decisionLog_effects_security      0
decisionLog_effects_morale        0
dtype: int64


In [ ]:
# Check for duplicates
print("Exact duplicate rows:")
print(master_df.duplicated().sum())

Exact duplicate rows:
0


In [ ]:
# Check for duplicates based on run_id and decision_order
decision_key = ["run_id", "decision_order"]
print("Duplicate run_id + decision_order rows:")
print(master_df.duplicated(subset=decision_key).sum())

Duplicate run_id + decision_order rows:
0


In [ ]:
# Summary of unique runs and total rows
print("Unique runs:")
print(master_df["run_id"].nunique())
print()

print("Total rows:")
print(len(master_df))
print()

run_summary = master_df[["run_id", "n_decisions"]].drop_duplicates()
print("Sum of n_decisions across unique runs:")
print(run_summary["n_decisions"].sum())

Unique runs:
137

Total rows:
2024

Sum of n_decisions across unique runs:
2024


In [ ]:
# Distribution of runs and rows by source folder
print("Runs by source folder:")
print(master_df[["run_id", "source_folder"]].drop_duplicates()["source_folder"].value_counts())
print()

print("Rows by source folder:")
print(master_df["source_folder"].value_counts())

Runs by source folder:
source_folder
CT4010 Morning Decision Logs      37
CT4010 afternoon decision logs    33
CT5056 Afternoon Decision Logs    28
CT5056 Morning Decision Logs      16
CT6033 Decision Logs              15
CT7074 Decision Logs               8
Name: count, dtype: int64

Rows by source folder:
source_folder
CT4010 Morning Decision Logs      540
CT4010 afternoon decision logs    487
CT5056 Afternoon Decision Logs    416
CT5056 Morning Decision Logs      238
CT6033 Decision Logs              224
CT7074 Decision Logs              119
Name: count, dtype: int64


In [13]:
# Check completed vs ended early
run_status = master_df[["run_id", "n_decisions", "completed_run", "ended_early"]].drop_duplicates()

print("Run status counts:")
print(run_status[["completed_run", "ended_early"]].value_counts())
print()

print("Distribution of n_decisions:")
print(run_status["n_decisions"].value_counts().sort_index())

Run status counts:
completed_run  ended_early
True           False          126
False          True            11
Name: count, dtype: int64

Distribution of n_decisions:
n_decisions
10      1
11      3
12      2
13      3
14      2
15    126
Name: count, dtype: int64


In [ ]:
# Check decision order consistency
decision_order_check = (
    master_df.groupby("run_id")["decision_order"]
    .agg(["min", "max", "nunique"])
    .reset_index()
)

decision_order_check["expected_ok"] = (
    (decision_order_check["min"] == 1) &
    (decision_order_check["max"] == decision_order_check["nunique"])
)

print("Decision order problems:")
print((~decision_order_check["expected_ok"]).sum())

decision_order_check[~decision_order_check["expected_ok"]].head(20)

Decision order problems:
0


,run_id,min,max,nunique,expected_ok


In [ ]:
# Check for blank strings in text columns
text_cols = [
    "decisionLog_scenario",
    "decisionLog_choice",
    "decisionLog_outcome"
]

for col in text_cols:
    blank_count = master_df[col].astype(str).str.strip().eq("").sum()
    print(f"{col} blank strings: {blank_count}")

decisionLog_scenario blank strings: 0
decisionLog_choice blank strings: 0
decisionLog_outcome blank strings: 0


In [ ]:
# Check effect columns are numeric
effect_cols = [
    "decisionLog_effects_budget",
    "decisionLog_effects_reputation",
    "decisionLog_effects_security",
    "decisionLog_effects_morale"
]

print(master_df[effect_cols].dtypes)
print()

print("Non-numeric check after coercion:")
for col in effect_cols:
    converted = pd.to_numeric(master_df[col], errors="coerce")
    print(col, "non-numeric values:", converted.isna().sum() - master_df[col].isna().sum())

decisionLog_effects_budget        int64
decisionLog_effects_reputation    int64
decisionLog_effects_security      int64
decisionLog_effects_morale        int64
dtype: object

Non-numeric check after coercion:
decisionLog_effects_budget non-numeric values: 0
decisionLog_effects_reputation non-numeric values: 0
decisionLog_effects_security non-numeric values: 0
decisionLog_effects_morale non-numeric values: 0


In [17]:
# check final score columns are numeric
final_score_cols = [
    "finalScores_budget",
    "finalScores_reputation",
    "finalScores_security",
    "finalScores_morale"
]

for col in final_score_cols:
    master_df[col] = pd.to_numeric(master_df[col], errors="coerce")

print(master_df[final_score_cols].dtypes)

finalScores_budget        int64
finalScores_reputation    int64
finalScores_security      int64
finalScores_morale        int64
dtype: object


In [ ]:
#check for GAME OVER in outcomes
game_over_rows = master_df[
    master_df["decisionLog_outcome"].astype(str).str.contains("GAME OVER", case=False, na=False)
]

print("Rows containing GAME OVER:")
print(len(game_over_rows))
print()

game_over_rows[[
    "run_id",
    "source_folder",
    "decision_order",
    "decisionLog_choice",
    "decisionLog_outcome"
]].head(10)

Rows containing GAME OVER:
17



,run_id,source_folder,decision_order,decisionLog_choice,decisionLog_outcome
190,run_1776088126013_280c4e5e43f4e,CT4010 afternoon decision logs,11,Replace the Server Now,<strong>Security:</strong> increases because o...
295,run_1776088351584_3619ce98924cc,CT4010 afternoon decision logs,15,Replace the Server Now,<strong>Security:</strong> increases because o...
353,run_1776088425352_64296be77475,CT4010 afternoon decision logs,13,Hybrid Storage Model,<strong>Security:</strong> increases moderatel...
366,run_1776088472457_0c98f96a50beb8,CT4010 afternoon decision logs,13,Replace the Server Now,<strong>Security:</strong> increases because o...
648,run_1776075463314_1797effd5d488,CT4010 Morning Decision Logs,12,Hybrid Storage Model,<strong>Security:</strong> increases moderatel...
975,run_1776076382956_e9df293cff1178,CT4010 Morning Decision Logs,12,Monitoring with Consent and Safeguards,<strong>Security:</strong> increases moderatel...
1016,run_1776076599533_aefc715071823,CT4010 Morning Decision Logs,11,Hybrid Storage Model,<strong>Security:</strong> increases moderatel...
1026,run_1776076634851_8bdcf33b3f7478,CT4010 Morning Decision Logs,10,Refuse Payment and Restore from Backups,<strong>Security:</strong> increases because s...
1037,run_1776174592230_b6cf3ad01c9778,CT5056 Afternoon Decision Logs,11,Strengthen On-Premises Storage,<strong>Security:</strong> increases because l...
1052,run_1776174603797_26b878f0f0307,CT5056 Afternoon Decision Logs,15,Replace the Server Now,<strong>Security:</strong> increases because o...


In [19]:
# final summary of key columns and distributions
print("Unique scenarios:")
print(master_df["decisionLog_scenario"].nunique())
print()

print("Unique choices:")
print(master_df["decisionLog_choice"].nunique())
print()

print("Session times:")
print(master_df["session_time"].value_counts(dropna=False))
print()

print("Module codes:")
print(master_df["module_code"].value_counts(dropna=False))

Unique scenarios:
15

Unique choices:
45

Session times:
session_time
Afternoon        903
Morning          778
Not specified    343
Name: count, dtype: int64

Module codes:
module_code
CT4010    1027
CT5056     654
CT6033     224
CT7074     119
Name: count, dtype: int64


**Cleaning and Standardisation**

In [21]:
import html

# make a copy from the already loaded master dataframe
cleaned_df = master_df.copy()

def clean_text(text):
    if pd.isna(text):
        return text

    text = html.unescape(str(text))
    text = text.replace("\r\n", " ").replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_outcome_text(text):
    if pd.isna(text):
        return text

    text = html.unescape(str(text))

    # replace html line breaks with readable separators
    text = re.sub(r"(?i)<br\s*/?>", " | ", text)

    # remove all remaining html tags
    text = re.sub(r"<[^>]+>", "", text)

    # remove extra spaces and repeated separators
    text = re.sub(r"\s*\|\s*(\|\s*)+", " | ", text)
    text = re.sub(r"\s+", " ", text).strip(" | ").strip()

    return text

# clean text columns in place
cleaned_df["source_folder"] = cleaned_df["source_folder"].apply(clean_text)
cleaned_df["module_code"] = cleaned_df["module_code"].apply(clean_text)
cleaned_df["session_time"] = cleaned_df["session_time"].apply(clean_text)
cleaned_df["source_file"] = cleaned_df["source_file"].apply(clean_text)

cleaned_df["run_id"] = cleaned_df["run_id"].apply(clean_text)
cleaned_df["sent_at"] = cleaned_df["sent_at"].apply(clean_text)

cleaned_df["decisionLog_scenario"] = cleaned_df["decisionLog_scenario"].apply(clean_text)
cleaned_df["decisionLog_choice"] = cleaned_df["decisionLog_choice"].apply(clean_text)
cleaned_df["decisionLog_outcome"] = cleaned_df["decisionLog_outcome"].apply(clean_outcome_text)
cleaned_df["decisionLog_timestamp"] = cleaned_df["decisionLog_timestamp"].apply(clean_text)

# make sure numeric columns stay numeric
numeric_cols = [
    "finalScores_budget",
    "finalScores_reputation",
    "finalScores_security",
    "finalScores_morale",
    "decisionLog_effects_budget",
    "decisionLog_effects_reputation",
    "decisionLog_effects_security",
    "decisionLog_effects_morale",
    "decision_order",
    "n_decisions"
]

for col in numeric_cols:
    cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors="coerce")

# quick checks
print("Unique scenarios:", cleaned_df["decisionLog_scenario"].nunique())
print("Unique choices:", cleaned_df["decisionLog_choice"].nunique())
print()

print("First 3 cleaned outcomes:")
print(cleaned_df["decisionLog_outcome"].head(3))

# save cleaned csv
cleaned_file = P.decision_logs_cleaned_dir / "decision_logs_cleaned.csv"
cleaned_df.to_csv(cleaned_file, index=False)

print("\nCleaned file saved to:")
print(cleaned_file)
print("Shape:", cleaned_df.shape)

Unique scenarios: 15
Unique choices: 45

First 3 cleaned outcomes:
0    Security: remains stable because containment a...
1    Security: increases because cloud providers of...
2    Security: increases moderately because access ...
Name: decisionLog_outcome, dtype: object

Cleaned file saved to:
F:\scenario-based cyber security management simulation\ISMSimulationGame\ISMSimulationGame\Analysis\Data\Decision_Logs\Cleaned\decision_logs_cleaned.csv
Shape: (2024, 22)
